# 02 · A Árvore de Decisão Quântica Híbrida (HQDT)

Uma árvore de decisão normal escolhe cada corte (feature `f`, limiar `t`) pelo **ganho de informação (IG)**. A HQDT mistura o IG com uma **distância quântica**:

$$S(f, t) = \alpha \cdot IG + (1 - \alpha) \cdot d_q$$

- $d_q = 1 - F$, onde $F = |\langle \psi(x_1) | \psi(x_2) \rangle|^2$ é a **fidelidade** entre dois estados quânticos
- $x_1$ = média dos pontos que vão pra esquerda no corte, $x_2$ = centróide do nó
- os pontos viram estados quânticos pelo **ZZFeatureMap** (3 qubits)
- limiares candidatos: percentis 20, 40, 60 e 80 de cada feature
- $\alpha = 0.5$ no artigo

A ideia: preferir cortes que, além de separar bem as classes, criem uma partição "longe" do centro do nó no espaço de Hilbert.

In [1]:
import sys
sys.path.insert(0, "../src")  # pra importar o pacote hqdt sem instalar

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

# paleta categórica validada (daltonismo) + estilo discreto
CORES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e6e6e3", "grid.linewidth": 0.8,
    "axes.edgecolor": "#8a8a85", "axes.labelcolor": "#3b3b38",
    "xtick.color": "#5c5c58", "ytick.color": "#5c5c58",
    "lines.linewidth": 2, "lines.markersize": 8, "figure.dpi": 110,
})

## 1. O circuito: ZZFeatureMap

Cada uma das 3 features do PCA controla rotações em um qubit, e as portas CNOT + fase criam emaranhamento entre os pares de qubits.

In [2]:
from qiskit.circuit.library import zz_feature_map

fm = zz_feature_map(feature_dimension=3, reps=1)
fm.decompose().draw("text", fold=120)

┌────────────┐┌───────────────┐                                                   »
0: ┤ U(π/2,0,π) ├┤ U(0,0,2*x[0]) ├──■────────────────────────────────────────■────■──»
   ├────────────┤├───────────────┤┌─┴─┐┌──────────────────────────────────┐┌─┴─┐  │  »
1: ┤ U(π/2,0,π) ├┤ U(0,0,2*x[1]) ├┤ X ├┤ U(0,0,(-π + x[0])*(-π + x[1])*2) ├┤ X ├──┼──»
   ├────────────┤├───────────────┤└───┘└──────────────────────────────────┘└───┘┌─┴─┐»
2: ┤ U(π/2,0,π) ├┤ U(0,0,2*x[2]) ├──────────────────────────────────────────────┤ X ├»
   └────────────┘└───────────────┘                                              └───┘»
«                                                                                          
«0: ──────────────────────────────────────■────────────────────────────────────────────────
«                                         │                                                
«1: ──────────────────────────────────────┼────■────────────────────────────────────────■──
«   ┌──────────────────────────────────┐┌─┴─┐┌─┴─┐┌──────────────────────────────────┐┌─┴─┐
«2: ┤ U(0,0,(-π + x[0])*(-π + x[2])*2) ├┤ X ├┤ X ├┤ U(0,0,(-π + x[1])*(-π + x[2])*2) ├┤ X ├
«   └──────────────────────────────────┘└───┘└───┘└──────────────────────────────────┘└───┘

## 2. Fidelidade entre dois pontos

Fidelidade 1 = estados iguais. Quanto mais diferentes os pontos, mais a fidelidade cai, e mais a distância $d_q = 1 - F$ sobe.

In [3]:
from qiskit_machine_learning.kernels import FidelityQuantumKernel

kernel = FidelityQuantumKernel(feature_map=fm)
a = np.array([[0.0, 0.0, 0.0]])
for delta in [0.0, 0.1, 0.5, 1.0, 2.0]:
    b = a + delta
    F = kernel.evaluate(a, b)[0][0]
    print(f"distância euclidiana {np.linalg.norm(b - a):.2f} -> fidelidade {F:.3f} -> d_q {1 - F:.3f}")

distância euclidiana 0.00 -> fidelidade 1.000 -> d_q 0.000
distância euclidiana 0.17 -> fidelidade 0.334 -> d_q 0.666
distância euclidiana 0.87 -> fidelidade 0.446 -> d_q 0.554
distância euclidiana 1.73 -> fidelidade 0.357 -> d_q 0.643
distância euclidiana 3.46 -> fidelidade 0.299 -> d_q 0.701


Repare que a relação **não é monotônica**: as rotações são periódicas, então pontos distantes podem "dar a volta" e parecer próximos. Isso é uma diferença real em relação a um kernel clássico como o RBF.

## 3. Treinando a HQDT

In [4]:
from hqdt import HybridQuantumDecisionTree, make_split

X_tr, X_te, y_tr, y_te = make_split(5000, seed=0)   # amostra estratificada + split 70/30
print("treino:", X_tr.shape, " teste:", X_te.shape)

hqdt = HybridQuantumDecisionTree(max_depth=4, alpha=0.5, distance="quantum")
hqdt.fit(X_tr, y_tr)

print(f"acurácia no teste: {hqdt.score(X_te, y_te):.3f}")
print(f"tempo de treino:   {hqdt.fit_time_:.2f} s")
print(f"profundidade: {hqdt.get_depth()}  | folhas: {hqdt.get_n_leaves()}")

treino: (3500, 3)  teste: (1500, 3)


acurácia no teste: 0.897
tempo de treino:   0.53 s
profundidade: 4  | folhas: 9


In [5]:
print(hqdt.print_tree(feature_names=["PC1", "PC2", "PC3"], class_names=["normal", "ataque"]))

PC2 <= -1.998
|   -> ataque
PC2 >  -1.998
|   PC2 <= 0.680
|   |   PC1 <= 1.914
|   |   |   PC1 <= 0.492
|   |   |   |   -> ataque
|   |   |   PC1 >  0.492
|   |   |   |   -> normal
|   |   PC1 >  1.914
|   |   |   PC2 <= -0.482
|   |   |   |   -> normal
|   |   |   PC2 >  -0.482
|   |   |   |   -> normal
|   PC2 >  0.680
|   |   PC1 <= -1.229
|   |   |   PC1 <= -2.386
|   |   |   |   -> ataque
|   |   |   PC1 >  -2.386
|   |   |   |   -> ataque
|   |   PC1 >  -1.229
|   |   |   PC1 <= 0.138
|   |   |   |   -> normal
|   |   |   PC1 >  0.138
|   |   |   |   -> normal


## 4. Quantas vezes o circuito roda? E o cache?

O artigo usa um cache de fidelidade que aproveita a simetria $F(a,b) = F(b,a)$. Vamos medir quantas vezes o kernel foi calculado e quantas vezes o cache foi usado.

In [6]:
print("cálculos de fidelidade:", hqdt.n_kernel_calls_)
print("acertos no cache:      ", hqdt.n_cache_hits_)

cálculos de fidelidade: 96
acertos no cache:       0


**O cache não acerta nenhuma vez.** O motivo é que a chave do cache é o par (média da esquerda, centróide), e esses vetores são diferentes em cada candidato de corte, porque são médias de subconjuntos diferentes. Então nenhum par se repete.

O custo quântico já é pequeno de qualquer jeito: são no máximo `3 features × 4 limiares = 12` fidelidades por nó, e até 15 nós internos com profundidade 4. O tempo de treino é dominado pela simulação desses ~100–170 circuitos, não pelo tamanho do dataset.

## 5. Comparação rápida com a árvore clássica do scikit-learn

In [7]:
from sklearn.tree import DecisionTreeClassifier

cart = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_tr, y_tr)
ig_only = HybridQuantumDecisionTree(max_depth=4, alpha=1.0, distance="none").fit(X_tr, y_tr)

pd.DataFrame({
    "acurácia": [cart.score(X_te, y_te), ig_only.score(X_te, y_te), hqdt.score(X_te, y_te)],
    "folhas": [cart.get_n_leaves(), ig_only.get_n_leaves(), hqdt.get_n_leaves()],
}, index=["sklearn (depth=4)", "só IG + percentis (alpha=1)", "HQDT (alpha=0.5)"]).round(3)

,acurácia,folhas
sklearn (depth=4),0.945,14
só IG + percentis (alpha=1),0.937,13
HQDT (alpha=0.5),0.897,9


Isso é uma única semente. A comparação de verdade, com vários tamanhos, 5 sementes e ablações, está no **notebook 03**.